# 🧬 Cuaderno 05 - Secuencias y k-mers

**Curso:**: Bioinformática
**Tema:**: Secuencias y k-mers
**Repositorio:**: `bio-notebooks`
**Herramientas:**: Python, Jupyter Notebook, Biopython

## Competencias del curso y del laboratorio
- Implementar algoritmos de alineamiento de secuencias global utilizando programación dinámica.
- Desarrollar algoritmos eficientes para el conteo y análisis de k-mers en secuencias biológicas, identificando patrones y visualizando información genómica.
- Aplicar conceptos fundamentales de la Bioinformática para analizar similitud entre secuencias biológicas y extraer información relevante de datos genómicos.
- Utilizar herramientas y bibliotecas de Python, como Biopython, para manipular y analizar secuencias biológicas, facilitando la interpretación de datos genómicos y la visualización de resultados.

## Contenido
1. Marco teórico
2. Implementación de algoritmos de conteo de k-mers
3. Análisis de frecuencias de k-mers en secuencias biológicas
4. Visualización de resultados y patrones de k-mers
5. Aplicaciones prácticas en genómica y metagenómica

## 1. Marco teórico
En esta sección se introduce el concepto de k-mers, su importancia en el análisis genómico y las aplicaciones prácticas en bioinformática. Se discuten las técnicas de conteo de k-mers, las estructuras de datos utilizadas para su almacenamiento y los desafíos computacionales asociados con el análisis de k-mers en grandes conjuntos de datos genómicos.

Los k-mers son subsecuencias contiguas de longitud k extraídas a partir de una secuencia biológica, ya sea ADN, ARN o proteínas. El análisis de k-mers constituye una de las técnicas fundamentales para el procesamiento y estudio de información genómica debido a su eficiencia computacional y su capacidad para identificar patrones biológicos relevantes.

Dada una secuencia de longitud n, el número total de k-mers que pueden extraerse se calcula mediante:
n − k + 1

Por ejemplo, para la secuencia ATGCA y k = 3, los k-mers obtenidos son:
- ATG
- TGC
- GCA

En secuencias de ADN existen cuatro nucleótidos posibles (A,T,C,G), por lo que el número total de combinaciones posibles de k-mers está dado por 4^k.

El análisis de frecuencias de k-mers permite identificar regiones repetitivas, patrones evolutivos y similitudes entre organismos. Además, constituye la base de múltiples aplicaciones modernas, incluyendo:
- ensamblaje de genomas
- alineamiento de secuencias
- metagenómica
- detección de mutaciones
- clasificación taxonómica
- búsqueda de similitud genética

Desde el punto de vista computacional, el conteo de k-mers requiere algoritmos eficientes debido al gran volumen de datos genómicos. Para ello, suelen emplearse estructuras de datos como tablas hash, diccionarios y árboles Trie, así como técnicas de optimización mediante ventanas deslizantes (sliding window) y paralelización.

En aplicaciones reales, los valores de k suelen variar entre 3 y 31 dependiendo del objetivo del análisis. Valores pequeños permiten detectar patrones generales, mientras que valores grandes ofrecen mayor especificidad genética, aunque incrementan el costo computacional y el uso de memoria.

### Funciones necesarias para el análisis de k-mers

Para llevar a cabo el análisis de k-mers, es necesario implementar funciones que permitan:
1. Descargar secuencias biológicas desde bases de datos como NCBI utilizando Biopython.
2. Leer secuencias desde archivos FASTA para su posterior análisis.

In [4]:
from Bio import Entrez, SeqIO
import os
from typing import Tuple, Optional

# Configuramos tnuestro email (NCBI exige un email valido)
# Entrez.email = "miemail@example.com"
# Si tenemos api_key (opcional), usamos: Entrez.api_key = "TU_API_KEY"

def fetch_fasta_from_ncbi(accession: str, email: str, save_path: Optional[str] = None, db: str = "nucleotide") -> Tuple[str, str, str]:
    """
    Descargamso un registro FASTA desde NCBI usando Biopython Entrez y nos devuelve (id, descripcion, secuencia).
    Si proporcioanmos save_path , guardamos el archivo FASTA en esa ruta
    Parametros:
      - accession: ID o accesión (por ejemplo 'NM_000518.5' o 'U14680.1')
      - email: el email es obligatorio para usar Entrez (NCBI)
      - save_path: si no usamos None, se guarda el archivo FASTA en la ruta uqe ponemos
      - db: 'nucleotide' o 'protein' según corresponda
    Retorna:
      (record.id, record.description, str(record.seq))
    """
    Entrez.email = email
    handle = None
    try:
        handle = Entrez.efetch(db=db, id=accession, rettype="fasta", retmode="text")
        # Parseamos con SeqIO para devolver valores consistentes
        record = SeqIO.read(handle, "fasta")
        seq_id = record.id
        desc = record.description
        seq = str(record.seq)
    finally:
        if handle is not None:
            handle.close()

    if save_path:
        # Nos aseguramos de que el directorio existe
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        # Escribimos el archio FASTA (usando SeqIO para formateo correcto)
        with open(save_path, "w") as outfh:
            SeqIO.write(record, outfh, "fasta")

    return seq_id, desc, seq

In [5]:
#Leer fasta
def leer_fasta(filepath: str) -> tuple[str, str, str]:
    """Leemos un archivo FASTA y nos devuelve el ID, la descripción y la secuencia
    Arguments:
        filepath (str): Es la ruta al archivo FASTA

    Returns:
        tuple[str, str, str]: ID, descripción y secuencia
    """
    with open(filepath, 'r') as file:
        lineas = file.readlines()
    header = lineas[0].strip()
    #print("header: ",header)
    #print("header.split(): ",header.split())
    secuencia_id = header.split()[0][1:]  # Eliminar el '>' y obtener el ID
    descripcion = ' '.join(header.split()[1:])  # Obtener la descripción
    secuencia = ''.join(linea.strip() for linea in lineas[1:]).upper()  # Unir las líneas de la secuencia
    return secuencia_id, descripcion, secuencia

In [ ]:
from dataclasses import dataclass
#Dataclass nos genera automaticamente nuestro constructor
@dataclass
class Secuencia:
    id: str
    descripcion: str
    secuencia: str

## 2. Implementación de algoritmos de conteo de k-mers
En esta sección se implementan algoritmos para el conteo de k-mers en secuencias biológicas utilizando Python. Se presentan diferentes enfoques, incluyendo el uso de diccionarios para almacenar las frecuencias de k-mers y técnicas de optimización para mejorar la eficiencia del conteo. Además, se discuten las ventajas y desventajas de cada método, así como su aplicabilidad en diferentes contextos genómicos.

### 2.1. Descarfa de secuencias desde NCBI utilizando Biopython
En esta subsección se muestra cómo descargar secuencias biológicas desde la base de datos NCBI utilizando la biblioteca Biopython. Se implementa una función que permite obtener secuencias en formato FASTA a partir de un ID de acceso, facilitando su posterior análisis de k-mers. Además, se discuten las consideraciones importantes al interactuar con la API de NCBI, como la gestión de solicitudes y el manejo de errores.

In [6]:
#Lista de accsession
# NM_000518.5, NM_009764
lista_accession = ["NM_000518.5", "NM_009764", "NC_000913.2", "NC_003197.2"]

#Ejemplo de uso de la función fetch_fasta_from_ncbi
path = "data/"
email = "rhyi888@gmail.com"  # Reemplazamos con nuestro email
for accession in lista_accession:
    save_path = path + accession + ".fasta"  # Guardamos cada secuencia con su accession como nombre de archivo
    seq_id, desc, seq = fetch_fasta_from_ncbi(accession=accession, email=email, save_path=save_path)
    # Descargamos la secuencia y la guardamos en un archivo FASTA
    print(f"Accession: {accession}")
    print(f"ID: {seq_id}")
    print(f"Descripción: {desc}")
    print(f"Secuencia: {seq[:50]}...")  # Imprimimos solo los primeros 50 caracteres de la secuencia para evitar saturar la salida

Accession: NM_000518.5
ID: NM_000518.5
Descripción: NM_000518.5 Homo sapiens hemoglobin subunit beta (HBB), mRNA
Secuencia: ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC...
Accession: NM_009764
ID: NM_009764.3
Descripción: NM_009764.3 Mus musculus breast cancer 1, early onset (Brca1), mRNA
Secuencia: GTTCCGAAAGGCTAGCGCTAGGCGCCAAGCGGCCGGTTTCCTTGGCGACG...
Accession: NC_000913.2
ID: NC_000913.2
Descripción: NC_000913.2 Escherichia coli str. K-12 substr. MG1655, complete genome
Secuencia: AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAA...
Accession: NC_003197.2
ID: NC_003197.2
Descripción: NC_003197.2 Salmonella enterica subsp. enterica serovar Typhimurium str. LT2, complete genome
Secuencia: AGAGATTACGTCTGGTTGCAAGAGATCATGACAGGGGGAATTGGTTGAAA...


### 2.2. Lectura de secuencias desde archivos FASTA
En esta subsección se muestra cómo leer secuencias biológicas desde archivos FASTA utilizando un implementacion propia. Se implementa una función que extrae el ID, la descripción y la secuencia de un archivo FASTA, facilitando su posterior análisis de k-mers. Además, se discuten las consideraciones importantes al manejar archivos FASTA, como la gestión de secuencias largas y la normalización de caracteres.

In [9]:
#Lista de secuecnias
nombres_secuencias = ["ejemplo_labo", "NM_000518.5", "NM_009764", "NC_000913.2", "NC_003197.2"]
path = 'data/'
for secuencia in nombres_secuencias:
    filepath = path + secuencia + ".fasta"
    secuencia_id, descripcion, secuencia = leer_fasta(filepath=filepath)

    #Impresion de resultados
    print(f"ID: {secuencia_id}")
    print(f"Descripción: {descripcion}")
    print(f"Secuencia: {secuencia[:50]}...")  # Imprimimos solo los primeros 50 caracteres de la secuencia para evitar saturar la salida

ID: ejemplo_labo.3
Descripción: Mus musculus breast cancer 1, early onset (Brca1), mRNA
Secuencia: ATGCA...
ID: NM_000518.5
Descripción: Homo sapiens hemoglobin subunit beta (HBB), mRNA
Secuencia: ACATTTGCTTCTGACACAACTGTGTTCACTAGCAACCTCAAACAGACACC...
ID: NM_009764.3
Descripción: Mus musculus breast cancer 1, early onset (Brca1), mRNA
Secuencia: GTTCCGAAAGGCTAGCGCTAGGCGCCAAGCGGCCGGTTTCCTTGGCGACG...
ID: NC_000913.2
Descripción: Escherichia coli str. K-12 substr. MG1655, complete genome
Secuencia: AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAA...
ID: NC_003197.2
Descripción: Salmonella enterica subsp. enterica serovar Typhimurium str. LT2, complete genome
Secuencia: AGAGATTACGTCTGGTTGCAAGAGATCATGACAGGGGGAATTGGTTGAAA...
